# Лабораторная работа №1
## Основы Python через анализ рыночной позиции

### Итог работы

Вы получите набор исходных строк из условной торговой системы,
преобразуете их в корректные типы, рассчитаете экономику позиции,
проверите риск-ограничения и сформируете текстовый отчёт. Все задачи
образуют один конвейер, а не набор несвязанных упражнений.

### Разрешённые средства

Переменные, скалярные типы, преобразование типов, арифметические и
логические операторы, методы строк, f-строки, `None` и условное
выражение `x if condition else y`. Циклы, функции, классы, NumPy и
Pandas пока не используются.

### Правило выполнения

Запускайте ячейки сверху вниз.



## 1. Состояние ноутбука и переменные

Jupyter хранит результаты выполненных ячеек. Если изменить значение
выше, но не перезапустить зависимые ячейки, на экране может остаться
устаревший результат. Поэтому перед сдачей используйте **Restart
Kernel and Run All Cells**.

Переменная — имя, связанное с объектом. Оператор `=` выполняет
присваивание, `==` — сравнение.


In [67]:
import sys

print("Python:", sys.version.split()[0])
position_state = "draft"
print("Начальное состояние:", position_state)


Python: 3.14.2
Начальное состояние: draft


### Задание 1. Паспорт выполнения

Создайте переменные:

- `student_name` — непустая строка с вашим именем;
- `lab_number` — целое число 1;
- `is_completed` — логическое `False`;
- `submitted_at` — `None`, так как работа ещё не сдана.

Выведите значения и их типы через `type()`.


In [68]:
student_name = "Владислав"  # впишите имя
lab_number = 1
is_completed = False
submitted_at = "not set"

# Выведите каждое значение и его тип
print(f'{student_name} {type(student_name)}\n{lab_number} {type(lab_number)}\n{is_completed} {type(is_completed)}\n{submitted_at} {type(submitted_at)}')

Владислав <class 'str'>
1 <class 'int'>
False <class 'bool'>
not set <class 'str'>


## 2. Граница системы: данные приходят строками

CSV, формы и API нередко передают числа как текст. Строка `"120"`
и число `120` — разные объекты: строку нельзя безопасно использовать
в арифметике до явного преобразования.


In [69]:
# Снимок позиции в том виде, в котором его прислала внешняя система.
raw_symbol = "  sber "
raw_quantity = "120"
raw_entry_price = "245.50"
raw_current_price = "258.20"
raw_dividend_per_share = "3.40"
raw_commission_percent = "0.05"  # именно проценты: 0.05%
raw_is_long = "yes"
raw_market_status = " OPEN "


### Задание 2. Приведение к рабочим типам

Создайте нормализованные значения:

- `symbol`: строка без внешних пробелов в верхнем регистре;
- `quantity`: `int`;
- цены и дивиденд: `float`;
- `commission_rate`: доля, то есть 0.0005, а не 0.05;
- `is_long`: результат сравнения нормализованной строки с `"yes"`;
- `market_status`: верхний регистр без внешних пробелов.

Исходные `raw_*` переменные не изменяйте.


In [70]:
symbol = raw_symbol.strip().upper()
quantity = int(raw_quantity)
entry_price = float(raw_entry_price)
current_price = float(raw_current_price)
dividend_per_share = float(raw_dividend_per_share)
commission_rate = float(raw_commission_percent)/100
is_long = raw_is_long == 'yes'
market_status = raw_market_status.upper().strip()
# YOUR CODE HERE

## 3. Числа, единицы измерения и порядок операций

В одном выражении легко смешать рубли, штуки и проценты. Перед
расчётом подпишите смысл переменных:

- `quantity × price` → рубли;
- `turnover × commission_rate` → рубли;
- `pnl / invested_capital` → безразмерная доля.

Скобки — часть модели, а не косметика. Для сравнения результатов
`float` используйте допустимую погрешность, а не всегда точное `==`.


### Задание 3. Экономика позиции

Рассчитайте:

1. `entry_turnover` — стоимость покупки без комиссии;
2. `entry_commission`;
3. `invested_capital` — покупка вместе с комиссией;
4. `current_value`;
5. `dividend_income`;
6. `gross_pnl` без дивидендов и комиссии;
7. `net_pnl` с дивидендами и комиссией;
8. `return_rate` относительно вложенного капитала;
9. `price_change_rate`;
10. `break_even_price` с учётом дивидендов и комиссии.


In [71]:
entry_turnover = entry_price * quantity
entry_commission = entry_price * commission_rate * quantity
invested_capital = entry_turnover + entry_commission
current_value = current_price * quantity
dividend_income = dividend_per_share * quantity
gross_pnl = current_value - entry_turnover
net_pnl = gross_pnl + dividend_income - entry_commission
return_rate = (net_pnl / invested_capital) * 100
price_change_rate = (current_price - entry_price)/entry_price * 100 #скорость и величина изменения актива за период
break_even_price = (entry_turnover + entry_commission)/quantity #цена безубыточности

print(entry_commission)
print(entry_turnover)
print(invested_capital)
print(gross_pnl)
print(net_pnl)
print(return_rate)
print(break_even_price)

14.73
29460.0
29474.73
1524.0
1917.27
6.504792410312156
245.62275


### Задание 4. Время торговой сессии

Время задано строками `HH:MM:SS`. Получите часы, минуты и секунды
срезами строки, переведите начало и конец в секунды от полуночи.
Рассчитайте длительность сессии, число сделок в час и средний оборот
в минуту. Сессия заканчивается в тот же день.


In [72]:
session_start = "09:50:00"
session_end = "18:50:00"
executed_trades = 427
session_turnover = 78_450_000.0


In [73]:
start_seconds = int(session_start[:2]) * 3600 + int(session_start[3:5]) * 60 + int(session_start[6:])
end_seconds = int(session_end[:2]) * 3600 + int(session_end[3:5]) * 60 + int(session_end[6:])
session_seconds = end_seconds - start_seconds
session_hours = session_seconds / 3600
trades_per_hour = executed_trades / session_hours
turnover_per_minute = trades_per_hour / 60
# YOUR CODE HERE
print(turnover_per_minute)

0.7907407407407407


## 4. Строки как часть интерфейса

Внутри программы данные должны иметь единый формат. На границе
системы из них создают человекочитаемые подписи и машинные ID.
`strip`, `lower`, `upper`, `title`, `split`, `join` и f-строки
покрывают большую часть задач первой обработки текста.


### Задание 5. Идентификаторы и подписи

Нормализуйте имя клиента, создайте email и идентификатор позиции.

- имя: `Ivan Sergeevich Petrov`;
- login: `ivan.petrov`;
- email: `ivan.petrov@example.edu`;
- instrument: `MOEX:TQBR:SBER`;
- position_id: `MOEX-TQBR-SBER-000120`.


In [74]:
raw_client_name = "  IVAN   sergeevich   PETROV  "
market = "moex"
board = "tqbr"
id = 120

client_name = " ".join(raw_client_name.title().split())
name_parts = raw_client_name.title().split()
login = str(name_parts[0]).lower() + '.' + str(name_parts[2]).lower()
email = login + '@example.edu'
instrument = f'{market.upper()}:{board.upper()}:SBER'
position_id = f'{market.upper()}-{board.upper()}-SBER-000{id}'
# YOUR CODE HERE
print(name_parts)

['Ivan', 'Sergeevich', 'Petrov']


## 5. Логика, ограничения и отсутствующие данные

Риск-решение складывается из независимых проверок. Храните каждую
проверку в отдельной переменной: так проще объяснить отказ. `None`
означает отсутствие значения, а не ноль и не пустую строку.


### Задание 6. Риск-фильтр

Проверки:

- стоимость позиции не выше 40 000 RUB;
- доходность не ниже −7%;
- среднедневной объём не ниже 500 000 бумаг;
- рынок открыт;
- метка времени риск-данных присутствует.

Сначала оставьте метку `None`. Решение должно быть `REFRESH_DATA`.
После этого задайте метку времени строкой и пересчитайте решение:
оно должно стать `ALLOW`.


In [75]:
max_position_value = 40_000.0
max_loss_rate = 0.07
min_daily_volume = 500_000
average_daily_volume = 1_250_000
risk_data_timestamp = None

position_limit_ok = None
loss_limit_ok = None
liquidity_ok = None
market_open = None
data_complete = None
risk_checks_pass = None
decision_without_data = ""

# После первого решения задайте непустую метку времени.
risk_data_timestamp = None
data_complete = None
final_risk_checks_pass = None
decision_with_data = ""

# YOUR CODE HERE
position_value = 30_000.0
loss_rate = 0.05
risk_data_timestamp = None
position_limit_ok = position_value <= max_position_value
loss_limit_ok = loss_rate >= -max_loss_rate
liquidity_ok = average_daily_volume >= min_daily_volume
market_open = True
data_complete = risk_data_timestamp is not None
final_risk_checks_pass = (position_limit_ok and loss_limit_ok and liquidity_ok and data_complete and market_open)
if risk_checks_pass is True:
    decision_without_data = "ALLOW"
else:
    decision_without_data = "RESFRESH DATA"
print(decision_without_data)

RESFRESH DATA


In [76]:
risk_data_timestamp = "20:48:00"
data_complete = risk_data_timestamp is not None
final_risk_checks_pass = (position_limit_ok and loss_limit_ok and liquidity_ok and data_complete and market_open)
if final_risk_checks_pass is True:
    decision_with_data = "ALLOW"
else:
    decision_with_data = "RESFRESH DATA"
print(decision_with_data)

ALLOW


## 6. Отладка: код выполняется не так, как задумано

Ошибка бывает синтаксической, типовой или логической. Последняя
опаснее: программа работает, но считает неверно. Прочитайте пять
проблем ниже и исправьте их, не меняя исходные `raw_*` значения.

### Задание 7. Исправление ошибок

1. `"120" * 3` повторяет строку, а не умножает число.
2. `0.05` процента ошибочно используют как долю 5%.
3. В имени остаются повторяющиеся пробелы.
4. `0.1 + 0.2 == 0.3` ненадёжно для float.
5. Отсутствие данных ошибочно проверяют через `== "None"`.


In [77]:
fixed_triple_quantity = None
fixed_commission = None
fixed_client_name = None
fixed_valus_equal = None
missing_value_detected = None
# YOUR CODE HERE

fixed_triple_quantity = int("120") * 3
fixed_commission = entry_turnover * 0.0005
fixed_client_name = " ".join(raw_client_name.title().split())
float_values_equal = round(0.1 + 0.2, 1) == round(0.3, 1)
missing_value_detected = missing_value_detected is None

In [78]:
assert fixed_triple_quantity == 360
assert abs(fixed_commission - 14.73) < 1e-9
assert fixed_client_name == "Ivan Sergeevich Petrov"
assert float_values_equal is True
assert missing_value_detected is True
print("Задание 7: OK")


Задание 7: OK


## 7. Итоговый мини-проект

Ниже находится новый, независимый снимок позиции. Не копируйте
готовые числа из предыдущих ячеек: повторите весь путь от строк до
решения.

### Задание 8. Позиция GAZP и стресс-сценарий

1. Преобразуйте исходные данные.
2. Рассчитайте вложенный капитал, текущую стоимость, чистый PnL и
   доходность.
3. Проверьте лимит позиции, максимальный убыток, ликвидность,
   статус рынка и наличие метки риска.
4. Сформируйте строку отчёта строго заданного формата.
5. Проведите стресс-сценарий с ценой 172 RUB и получите новое
   решение без изменения базового результата.


In [79]:
final_raw_symbol = " gazp "
final_raw_quantity = "350"
final_raw_entry_price = "181.40"
final_raw_current_price = "176.90"
final_raw_dividend = "0"
final_raw_commission_percent = "0.04"
final_market_status = "open"
final_raw_daily_volume = "850000"
final_risk_timestamp = "2026-03-01T10:05:00"

final_max_position_value = 70_000.0
final_max_loss_rate = 0.04
final_min_daily_volume = 500_000
stress_price = 172.0


In [80]:
final_symbol = final_raw_symbol.upper().strip()
final_quantity = int(final_raw_quantity)
final_entry_price = float(final_raw_entry_price)
final_current_price = float(final_raw_current_price)
final_dividend = int(final_raw_dividend)
final_commission_rate = float(final_raw_commission_percent) / 100
final_daily_volume = int(final_raw_daily_volume)

final_entry_turnover = final_entry_price * final_quantity
final_entry_commission = final_entry_price * final_commission_rate * final_quantity
final_invested_capital = final_entry_turnover + final_entry_commission
final_current_value = final_current_price * final_quantity
final_net_pnl = final_current_value - final_entry_turnover + final_dividend * final_quantity - final_entry_commission
final_return_rate = (final_net_pnl / final_invested_capital) * 100

final_position_ok = None
final_loss_ok = None
final_liquidity_ok = None
final_market_open = None
final_data_complete = None
final_allowed = None
final_decision = ""
final_report = ""

stress_current_value = None
stress_net_pnl = None
stress_return_rate = None
stress_allowed = None
stress_decision = ""

# YOUR CODE HERE
final_position_ok = final_current_value <= final_max_position_value
final_loss_ok = final_current_price / final_entry_price - 1 >= -final_max_loss_rate
final_liquidity_ok = final_daily_volume >= final_min_daily_volume
final_market_open = True
final_data_complete = final_risk_timestamp is not None
final_allowed = (final_position_ok and final_loss_ok and final_liquidity_ok and final_market_open and final_data_complete)
if final_allowed is True:
    final_decision = 'Allowed'
else:
    final_decision = 'Not allowed'
final_report = "Торговля"

stress_current_value = stress_price * final_quantity
stress_net_pnl = (stress_price - final_entry_price) * final_quantity + final_quantity * final_dividend - final_entry_commission
stress_return_rate = stress_net_pnl / final_invested_capital

stress_position_ok = stress_current_value <= final_max_position_value
stress_loss_ok = (stress_price / final_entry_price - 1) >= - final_max_loss_rate

stress_allowed = (stress_position_ok and stress_loss_ok and final_liquidity_ok and final_market_open and final_data_complete)
if stress_allowed is not True:
    stress_decision = 'Not allowed'
else:
    stress_decision = 'Allow'
final_report = ""

print(final_decision)
print(stress_decision)

Allowed
Not allowed


## Вывод и защита

В отдельной Markdown-ячейке после этой напишите 5–7 предложений:

1. Почему позиция с отрицательным PnL всё же получила ALLOW?
2. Какой именно лимит нарушен в stress-сценарии?
3. Где в расчётах использовались проценты, а где доли?
4. Какие два исходных значения сильнее всего влияют на решение?
5. Что станет опасным, если данные останутся строками?

### Контрольные вопросы

1. Чем `=` отличается от `==`?
2. Почему `"120" * 3` не равно 360?
3. Зачем делить процент комиссии на 100?
4. Чем `None` отличается от 0 и пустой строки?
5. Почему сравнение float иногда выполняют с допуском?
6. Как приоритет операций влияет на финансовую формулу?
7. Почему риск-проверки полезно хранить отдельно?
8. Что нужно сделать с ноутбуком перед сдачей?


Ответы на вопросы:
1) Позиция с отрицательным PnL получила allow, так как размер позиции не превосходит максимального размера позиции и средний дневной объем торгов больше минимального значения.
2) Стресс-сценарий нарушен из-за final_loss_ok, то есть потери больше максимально возможного процента потерь.
3) Проценты использовались при расчете комиссии, а доли - .
4) В основном на решение влияет цена и объем.
5) Будет невозможно корректно проводить между такими данными различные арифметические операции.

_Напишите здесь собственный вывод._
